# Trabalho Prático 05 — Agentes Inteligentes para Othello

**Disciplina:** GCC128 — Inteligência Artificial  
**Tecnologias:** Python 3, Pygame, LangChain, LangGraph e Groq  

O projeto implementa as regras do Othello, um agente heurístico e duas modalidades de autonomia para o agente LLM. Ambas compartilham ferramentas, memória, planejamento e estado do jogo. A interface Pygame e os experimentos comparativos serão desenvolvidos nas próximas etapas.

## 1. Dependências

Execute a célula abaixo **uma vez** no kernel escolhido. O comando `%pip` instala as bibliotecas no ambiente Python do próprio notebook. Recomenda-se utilizar um kernel associado a um ambiente virtual local.

In [1]:
%pip install -q pygame langchain langgraph langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Verificação do ambiente

Confere se as bibliotecas foram instaladas e podem ser importadas. Não acessa a API e não abre uma janela gráfica.

In [2]:
import sys
from importlib.metadata import version

import pygame
import langchain
import langgraph
import langchain_groq

print("Python:", sys.version.split()[0])
for pacote in ("pygame", "langchain", "langgraph", "langchain-groq"):
    print(f"{pacote}: {version(pacote)}")
print("Ambiente configurado. Nenhum agente foi executado.")

pygame 2.6.1 (SDL 2.28.4, Python 3.13.2)
Hello from the pygame community. https://www.pygame.org/contribute.html
Python: 3.13.2
pygame: 2.6.1
langchain: 1.4.9
langgraph: 1.2.20
langchain-groq: 1.1.3
Ambiente configurado. Nenhum agente foi executado.


## 3. Motor do Othello

Implementa o tabuleiro 8×8, as regras, a alternância de turnos e a detecção do fim da partida.

3.1 Inicialização do tabuleiro

In [3]:
# Representação das casas
VAZIO = 0
PRETO = 1
BRANCO = -1

TAMANHO = 8

def criar_tabuleiro():
    tabuleiro = [
        [VAZIO for _ in range(TAMANHO)]
        for _ in range(TAMANHO)
    ]

    # Posições iniciais das peças
    tabuleiro[3][3] = BRANCO
    tabuleiro[3][4] = PRETO
    tabuleiro[4][3] = PRETO
    tabuleiro[4][4] = BRANCO

    return tabuleiro


def exibir_tabuleiro(tabuleiro):
    simbolos = {
        VAZIO: ".",
        PRETO: "P",
        BRANCO: "B"
    }

    print("    " + " ".join(str(i) for i in range(TAMANHO)))

    for i, linha in enumerate(tabuleiro):
        print(f"{i}   " + " ".join(simbolos[peca] for peca in linha))

3.2 Verificação de jogadas válidas

In [4]:
# Oito direções possíveis: horizontal, vertical e diagonais
DIRECOES = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1)
]


def pecas_capturadas(tabuleiro, linha, coluna, jogador):
    if not (0 <= linha < TAMANHO and 0 <= coluna < TAMANHO):
        return []

    if tabuleiro[linha][coluna] != VAZIO:
        return []

    capturadas = []

    for dl, dc in DIRECOES:
        l, c = linha + dl, coluna + dc
        caminho = []

        # Percorre as peças adversárias nesta direção
        while (
            0 <= l < TAMANHO
            and 0 <= c < TAMANHO
            and tabuleiro[l][c] == -jogador
        ):
            caminho.append((l, c))
            l += dl
            c += dc

        # A sequência só é válida se terminar em uma peça própria
        if (
            caminho
            and 0 <= l < TAMANHO
            and 0 <= c < TAMANHO
            and tabuleiro[l][c] == jogador
        ):
            capturadas.extend(caminho)

    return capturadas


def movimentos_validos(tabuleiro, jogador):
    movimentos = []

    for linha in range(TAMANHO):
        for coluna in range(TAMANHO):
            if pecas_capturadas(tabuleiro, linha, coluna, jogador):
                movimentos.append((linha, coluna))

    return movimentos

3.3 Executar jogadas e inverter peças

In [5]:
def fazer_jogada(tabuleiro, linha, coluna, jogador):
    # Identifica as peças que serão capturadas
    capturadas = pecas_capturadas(
        tabuleiro, linha, coluna, jogador
    )

    # Impede movimentos inválidos
    if not capturadas:
        return False

    # Coloca a peça do jogador
    tabuleiro[linha][coluna] = jogador

    # Inverte as peças adversárias capturadas
    for l, c in capturadas:
        tabuleiro[l][c] = jogador

    return True

3.4 Controle de turnos e passagem de vez

In [6]:
def proximo_turno(tabuleiro, jogador_atual):
    adversario = -jogador_atual

    # Verifica se o adversário pode jogar
    if movimentos_validos(tabuleiro, adversario):
        return adversario

    # Se o adversário não puder, o jogador atual continua
    if movimentos_validos(tabuleiro, jogador_atual):
        return jogador_atual

    # Nenhum jogador possui movimentos válidos
    return None

3.5 Encerramento do jogo e contagem de peças

In [7]:
def contar_pecas(tabuleiro):
    pretas = sum(linha.count(PRETO) for linha in tabuleiro)
    brancas = sum(linha.count(BRANCO) for linha in tabuleiro)

    return pretas, brancas


def jogo_terminou(tabuleiro):
    return (
        not movimentos_validos(tabuleiro, PRETO)
        and not movimentos_validos(tabuleiro, BRANCO)
    )


def obter_vencedor(tabuleiro):
    if not jogo_terminou(tabuleiro):
        return None

    pretas, brancas = contar_pecas(tabuleiro)

    if pretas > brancas:
        return PRETO
    elif brancas > pretas:
        return BRANCO
    else:
        return VAZIO  # Empate

## 4. Ferramentas do jogo

As quatro ferramentas expõem leitura do tabuleiro, movimentos válidos, execução de jogada e situação da partida.

In [8]:
def iniciar_partida():
    return {
        "tabuleiro": criar_tabuleiro(),
        "jogador_atual": PRETO,
        "historico": []
    }


# Estado compartilhado entre as ferramentas
estado_jogo = iniciar_partida()

### 4.1 — Ferramentas de consulta do jogo

In [9]:
from langchain_core.tools import tool


@tool
def get_board_state() -> dict:
    """Consulta o estado atual do tabuleiro de Othello."""

    tabuleiro = estado_jogo["tabuleiro"]
    pretas, brancas = contar_pecas(tabuleiro)

    return {
        "board": [linha.copy() for linha in tabuleiro],
        "current_player": estado_jogo["jogador_atual"],
        "black_count": pretas,
        "white_count": brancas
    }


@tool
def get_valid_moves() -> dict:
    """Consulta os movimentos válidos do jogador atual.
    As coordenadas de linha e coluna começam em zero.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {"current_player": None, "moves": []}

    movimentos = movimentos_validos(
        estado_jogo["tabuleiro"], jogador
    )

    return {
        "current_player": jogador,
        "moves": [
            {"row": linha, "col": coluna}
            for linha, coluna in movimentos
        ]
    }

### 4.2 — Ferramenta make_move()

In [10]:
@tool
def make_move(row: int, col: int) -> dict:
    """Executa uma jogada de Othello para o jogador atual.
    As coordenadas row e col devem estar entre 0 e 7.
    """

    tabuleiro = estado_jogo["tabuleiro"]
    jogador = estado_jogo["jogador_atual"]

    # Verifica se a partida já terminou
    if jogador is None:
        return {
            "success": False,
            "message": "A partida já terminou."
        }

    # Executa a jogada
    if not fazer_jogada(tabuleiro, row, col, jogador):
        return {
            "success": False,
            "message": "Jogada inválida.",
            "valid_moves": movimentos_validos(tabuleiro, jogador)
        }

    # Registra a jogada no histórico
    estado_jogo["historico"].append({
        "player": jogador,
        "row": row,
        "col": col
    })

    # Atualiza o jogador atual
    proximo = proximo_turno(tabuleiro, jogador)
    estado_jogo["jogador_atual"] = proximo

    return {
        "success": True,
        "message": "Jogada realizada com sucesso.",
        "next_player": proximo,
        "game_over": proximo is None
    }

### 4.3 — Ferramenta get_game_status()

In [11]:
@tool
def get_game_status() -> dict:
    """Consulta o estado da partida de Othello, incluindo
    pontuação, jogador atual, encerramento e vencedor.
    """

    tabuleiro = estado_jogo["tabuleiro"]

    pretas, brancas = contar_pecas(tabuleiro)
    terminou = jogo_terminou(tabuleiro)

    return {
        "game_over": terminou,
        "current_player": estado_jogo["jogador_atual"],
        "black_count": pretas,
        "white_count": brancas,
        "total_moves": len(estado_jogo["historico"]),
        "winner": obter_vencedor(tabuleiro)
    }

## 5. Agente heurístico

5.1 — Definir a heurística de avaliação do tabuleiro.

In [12]:
# Pesos estratégicos das posições do tabuleiro
PESOS_POSICAO = [
    [100, -20,  10,   5,   5,  10, -20, 100],
    [-20, -50,  -2,  -2,  -2,  -2, -50, -20],
    [ 10,  -2,   5,   1,   1,   5,  -2,  10],
    [  5,  -2,   1,   0,   0,   1,  -2,   5],
    [  5,  -2,   1,   0,   0,   1,  -2,   5],
    [ 10,  -2,   5,   1,   1,   5,  -2,  10],
    [-20, -50,  -2,  -2,  -2,  -2, -50, -20],
    [100, -20,  10,   5,   5,  10, -20, 100]
]

# Quanto mais alto o valor, mais estratégica é a posição, e quanto mais baixo, mais arriscada é a posição.

In [13]:
# Avalia o tabuleiro de Othello para um jogador específico, atribuindo uma pontuação com base na posição das peças.
def avaliar_tabuleiro(tabuleiro, jogador):
    pontuacao = 0

    for linha in range(TAMANHO):
        for coluna in range(TAMANHO):

            peca = tabuleiro[linha][coluna]

            if peca == jogador:
                pontuacao += PESOS_POSICAO[linha][coluna]

            elif peca == -jogador:
                pontuacao -= PESOS_POSICAO[linha][coluna]

    return pontuacao

5.2 — Simular e avaliar possíveis jogadas.

In [14]:
def avaliar_jogadas(tabuleiro, jogador):
    resultados = []

    for linha, coluna in movimentos_validos(tabuleiro, jogador):

        # Cria uma cópia independente do tabuleiro
        simulacao = [l.copy() for l in tabuleiro]

        # Simula a jogada e a captura das peças
        fazer_jogada(simulacao, linha, coluna, jogador)

        # Avalia o tabuleiro após a jogada
        pontuacao = avaliar_tabuleiro(simulacao, jogador)

        resultados.append({
            "row": linha,
            "col": coluna,
            "score": pontuacao
        })

    return resultados

5.3 — Implementar a escolha automática da melhor jogada.

In [15]:
def escolher_melhor_jogada(tabuleiro, jogador):
    jogadas = avaliar_jogadas(tabuleiro, jogador)

    # Não existem movimentos válidos
    if not jogadas:
        return None

    # Seleciona a jogada com maior pontuação
    melhor_jogada = jogadas[0]

    for jogada in jogadas:
        if jogada["score"] > melhor_jogada["score"]:
            melhor_jogada = jogada

    return melhor_jogada

### 5.4 — Partida entre agentes heurísticos

A função abaixo permite executar uma partida completa de teste quando chamada explicitamente.

In [16]:
def jogar_partida_heuristica():
    global estado_jogo
    estado_jogo = iniciar_partida()

    while estado_jogo["jogador_atual"] is not None:
        tabuleiro = estado_jogo["tabuleiro"]
        jogador = estado_jogo["jogador_atual"]
        jogada = escolher_melhor_jogada(tabuleiro, jogador)

        if jogada is None:
            raise ValueError("Jogador atual sem movimentos válidos.")

        resultado = make_move.invoke({
            "row": jogada["row"],
            "col": jogada["col"]
        })

        if not resultado["success"]:
            raise RuntimeError(resultado["message"])

    return get_game_status.invoke({})

## 6. Integração da LLM com Groq e LangGraph

### Modos de autonomia

**1 — Estratégica:** o LangGraph define a sequência e a LLM escolhe uma jogada estruturada.  
**2 — Operacional:** a LLM escolhe quais ferramentas chamar; o LangGraph limita o ciclo, valida chamadas e encerra após uma jogada válida.

A escolha abaixo controla os turnos seguintes. Execute a célula novamente para trocar de modo.

In [17]:
escolha = input(
    "Modo de autonomia (1 = estratégica, 2 = operacional): "
).strip()

if escolha not in ("1", "2"):
    raise ValueError("Escolha 1 (estratégica) ou 2 (operacional).")

MODO_AUTONOMIA = {
    "1": "estrategica",
    "2": "operacional"
}[escolha]

print("Modo selecionado:", MODO_AUTONOMIA)

Modo selecionado: operacional


### 6.1 — Configurar a API Groq e decisões estruturadas

A chave é solicitada em tempo de execução somente se a variável de ambiente GROQ_API_KEY ainda não estiver configurada. Ela não fica gravada no notebook.

In [18]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")

In [19]:
from langchain_groq import ChatGroq
from pydantic import BaseModel, ConfigDict, Field

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


class DecisaoOthello(BaseModel):
    model_config = ConfigDict(extra="forbid")

    row: int = Field(description="Linha da jogada, entre 0 e 7")
    col: int = Field(description="Coluna da jogada, entre 0 e 7")
    justificativa: str = Field(description="Motivo estratégico resumido")


llm_decisor = llm.with_structured_output(
    DecisaoOthello,
    method="json_schema",
    strict=True
)

### 6.2 — Disponibilizar as ferramentas do Othello

In [20]:
# Ferramentas usadas diretamente pelo fluxo do agente
ferramentas = {
    "get_board_state": get_board_state,
    "get_valid_moves": get_valid_moves,
    "make_move": make_move,
    "get_game_status": get_game_status
}

### 6.3 — Fluxo do agente com LangGraph

Cada modalidade possui um grafo próprio; ambas compartilham validação e execução de jogadas. No modo operacional, o ciclo permite até seis chamadas individuais, rejeita respostas com chamadas paralelas e encerra após uma jogada válida.

In [21]:
import json

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, ToolMessage
)


class EstadoAgente(MessagesState, total=False):
    observacao: dict
    historico: str
    planejamento: list
    decisao: dict
    origem_decisao: str
    erro_decisao: str
    jogada_escolhida: dict
    resultado: dict
    historico_inicial: int
    chamadas_ferramenta: int
    ferramentas_executadas: list


INSTRUCOES_ESTRATEGICAS = """Você joga Othello. Escolha uma das jogadas válidas fornecidas.
Considere os cantos, os pesos posicionais, a mobilidade e as respostas possíveis.
As coordenadas começam em zero. Retorne a decisão no formato estruturado solicitado."""


def observar(estado: EstadoAgente) -> dict:
    # O fluxo estratégico consulta as três ferramentas de observação.
    tabuleiro = ferramentas["get_board_state"].invoke({})
    movimentos = ferramentas["get_valid_moves"].invoke({})
    situacao = ferramentas["get_game_status"].invoke({})
    ferramentas_consultadas = [
        "get_board_state", "get_valid_moves", "get_game_status"
    ]

    if situacao["game_over"]:
        return {
            "observacao": {
                "board": tabuleiro["board"],
                "current_player": tabuleiro["current_player"],
                "moves": movimentos["moves"],
                "status": situacao
            },
            "historico": obter_historico(),
            "ferramentas_executadas": ferramentas_consultadas
        }

    jogador = tabuleiro["current_player"]
    if jogador not in (PRETO, BRANCO) or movimentos["current_player"] != jogador:
        raise RuntimeError("Estado inconsistente entre as ferramentas.")

    if not movimentos["moves"]:
        raise RuntimeError("O jogador atual não possui jogadas válidas.")

    return {
        "observacao": {
            "board": tabuleiro["board"],
            "current_player": jogador,
            "moves": movimentos["moves"],
            "status": situacao
        },
        "historico": obter_historico(),
        "ferramentas_executadas": ferramentas_consultadas
    }


def planejar(estado: EstadoAgente) -> dict:
    observacao = estado["observacao"]
    jogador = observacao["current_player"]

    if observacao["status"]["game_over"]:
        return {"planejamento": []}

    analises = planejar_jogadas(observacao["board"], jogador)
    if len(analises) != len(observacao["moves"]):
        raise RuntimeError("O planejamento não corresponde às jogadas válidas.")

    return {"planejamento": analises}


def decidir_estrategico(estado: EstadoAgente) -> dict:
    observacao = estado["observacao"]
    if observacao["status"]["game_over"]:
        return {"decisao": {}, "origem_decisao": "sem_jogada"}

    contexto = {
        "jogador": observacao["current_player"],
        "tabuleiro": observacao["board"],
        "jogadas_validas": observacao["moves"],
        "historico": estado["historico"],
        "planejamento": estado["planejamento"]
    }

    try:
        resposta = llm_decisor.invoke([
            SystemMessage(content=INSTRUCOES_ESTRATEGICAS),
            HumanMessage(content=json.dumps(contexto, ensure_ascii=False))
        ])
        if not isinstance(resposta, DecisaoOthello):
            resposta = DecisaoOthello.model_validate(resposta)
        return {
            "decisao": resposta.model_dump(),
            "origem_decisao": "llm",
            "erro_decisao": ""
        }
    except Exception as erro:
        return {
            "decisao": {},
            "origem_decisao": "fallback_heuristico",
            "erro_decisao": type(erro).__name__
        }


def validar_decisao(estado: EstadoAgente) -> dict:
    observacao = estado["observacao"]
    if observacao["status"]["game_over"]:
        return {"jogada_escolhida": {}}

    legais = observacao["moves"]
    decisao = estado.get("decisao", {})
    escolhida = next(
        (
            movimento for movimento in legais
            if movimento["row"] == decisao.get("row")
            and movimento["col"] == decisao.get("col")
        ),
        None
    )

    if escolhida is None:
        planejamento = estado["planejamento"]
        if planejamento:
            melhor = max(
                planejamento,
                key=lambda item: (
                    item["worst_response"],
                    item["score_now"],
                    -item["row"],
                    -item["col"]
                )
            )
            escolhida = {"row": melhor["row"], "col": melhor["col"]}
        else:
            escolhida = legais[0]

        return {
            "jogada_escolhida": escolhida,
            "origem_decisao": "fallback_heuristico",
            "erro_decisao": estado.get("erro_decisao") or "jogada_invalida"
        }

    return {"jogada_escolhida": escolhida}


def executar_jogada(estado: EstadoAgente) -> dict:
    if estado["observacao"]["status"]["game_over"]:
        return {"resultado": {"success": True, "game_over": True}}

    jogada = estado["jogada_escolhida"]
    jogador = estado["observacao"]["current_player"]
    quantidade_anterior = len(estado_jogo["historico"])

    resultado = ferramentas["make_move"].invoke({
        "row": jogada["row"],
        "col": jogada["col"]
    })

    if not resultado["success"]:
        raise RuntimeError(resultado["message"])

    if len(estado_jogo["historico"]) != quantidade_anterior + 1:
        raise RuntimeError("O turno deve registrar exatamente uma jogada.")

    registrada = estado_jogo["historico"][-1]
    if (
        registrada["player"] != jogador
        or registrada["row"] != jogada["row"]
        or registrada["col"] != jogada["col"]
    ):
        raise RuntimeError("A jogada registrada não corresponde à decisão.")

    usadas = estado.get("ferramentas_executadas", []) + ["make_move"]
    return {"resultado": resultado, "ferramentas_executadas": usadas}


# Modo 1: o grafo controla a ordem completa das etapas.
grafo_estrategico = StateGraph(EstadoAgente)
grafo_estrategico.add_node("observar", observar)
grafo_estrategico.add_node("planejar", planejar)
grafo_estrategico.add_node("decidir", decidir_estrategico)
grafo_estrategico.add_node("validar", validar_decisao)
grafo_estrategico.add_node("executar", executar_jogada)

grafo_estrategico.add_edge(START, "observar")
grafo_estrategico.add_edge("observar", "planejar")
grafo_estrategico.add_edge("planejar", "decidir")
grafo_estrategico.add_edge("decidir", "validar")
grafo_estrategico.add_edge("validar", "executar")
grafo_estrategico.add_edge("executar", END)

app_estrategico = grafo_estrategico.compile()


INSTRUCOES_OPERACIONAIS = """Você joga Othello e pode escolher quando usar as ferramentas.
Use as informações de memória e planejamento recebidas para avaliar a jogada.
As ferramentas get_board_state, get_valid_moves e get_game_status consultam o ambiente.
A ferramenta make_move executa uma jogada para o jogador atual; nunca escolha o adversário.
As coordenadas começam em zero. Faça uma chamada de ferramenta por resposta e pare
depois que uma chamada make_move for executada com sucesso. Não altere o tabuleiro
diretamente. Se uma ferramenta retornar erro, corrija a solicitação ou use as
informações recebidas. O sistema valida cada jogada e encerra o turno após um movimento."""


llm_com_ferramentas = llm.bind_tools(list(ferramentas.values()))
MAX_FERRAMENTAS_POR_TURNO = 6
no_ferramentas = ToolNode(
    list(ferramentas.values()),
    handle_tool_errors=True
)


def preparar_operacional(estado: EstadoAgente) -> dict:
    tabuleiro = [linha.copy() for linha in estado_jogo["tabuleiro"]]
    jogador = estado_jogo["jogador_atual"]
    movimentos = movimentos_validos(tabuleiro, jogador)

    if jogador not in (PRETO, BRANCO) or not movimentos:
        raise RuntimeError("Não há um turno válido para o agente.")

    jogadas = [
        {"row": linha, "col": coluna}
        for linha, coluna in movimentos
    ]
    analises = planejar_jogadas(tabuleiro, jogador)
    historico = obter_historico()
    contexto = {
        "jogador_atual": jogador,
        "tabuleiro": tabuleiro,
        "jogadas_validas": jogadas,
        "historico": historico,
        "planejamento": analises
    }

    return {
        "observacao": {
            "board": tabuleiro,
            "current_player": jogador,
            "moves": jogadas,
            "status": {"game_over": False}
        },
        "historico": historico,
        "planejamento": analises,
        "historico_inicial": len(estado_jogo["historico"]),
        "chamadas_ferramenta": 0,
        "erro_decisao": "",
        "ferramentas_executadas": [],
        "messages": [
            SystemMessage(content=INSTRUCOES_OPERACIONAIS),
            HumanMessage(content=json.dumps(contexto, ensure_ascii=False))
        ]
    }


def decidir_operacional(estado: EstadoAgente) -> dict:
    try:
        resposta = llm_com_ferramentas.invoke(estado["messages"])
    except Exception as erro:
        return {
            "erro_decisao": type(erro).__name__,
            "origem_decisao": "fallback_heuristico"
        }

    chamadas = resposta.tool_calls or []
    total = estado.get("chamadas_ferramenta", 0) + len(chamadas)
    erro_chamada = ""

    if not chamadas:
        erro_chamada = "resposta_sem_chamada_de_ferramenta"
    elif len(chamadas) != 1:
        erro_chamada = "multiplas_chamadas_de_ferramenta"
    elif chamadas[0]["name"] not in ferramentas:
        erro_chamada = "ferramenta_desconhecida"
    elif total > MAX_FERRAMENTAS_POR_TURNO:
        erro_chamada = "limite_de_ferramentas"

    return {
        "messages": [resposta],
        "chamadas_ferramenta": total,
        "erro_decisao": erro_chamada,
        "origem_decisao": (
            "fallback_heuristico" if erro_chamada else ""
        )
    }


def rotear_operacional(estado: EstadoAgente) -> str:
    if len(estado_jogo["historico"]) > estado["historico_inicial"]:
        return END
    if estado.get("erro_decisao"):
        return "fallback"
    return "tools"


def rotear_apos_ferramenta(estado: EstadoAgente) -> str:
    if len(estado_jogo["historico"]) > estado["historico_inicial"]:
        return END
    if estado["chamadas_ferramenta"] >= MAX_FERRAMENTAS_POR_TURNO:
        return "fallback"
    return "decidir"


def fallback_operacional(estado: EstadoAgente) -> dict:
    return {
        "decisao": {},
        "origem_decisao": "fallback_heuristico",
        "erro_decisao": (
            estado.get("erro_decisao") or "limite_de_ferramentas"
        )
    }


# Modo 2: a LLM escolhe as ferramentas; o grafo valida e limita o ciclo.
grafo_operacional = StateGraph(EstadoAgente)
grafo_operacional.add_node("preparar", preparar_operacional)
grafo_operacional.add_node("decidir", decidir_operacional)
grafo_operacional.add_node("tools", no_ferramentas)
grafo_operacional.add_node("fallback", fallback_operacional)
grafo_operacional.add_node("validar", validar_decisao)
grafo_operacional.add_node("executar", executar_jogada)

grafo_operacional.add_edge(START, "preparar")
grafo_operacional.add_edge("preparar", "decidir")
grafo_operacional.add_conditional_edges(
    "decidir",
    rotear_operacional,
    {"tools": "tools", "fallback": "fallback", END: END}
)
grafo_operacional.add_conditional_edges(
    "tools",
    rotear_apos_ferramenta,
    {"decidir": "decidir", "fallback": "fallback", END: END}
)
grafo_operacional.add_edge("fallback", "validar")
grafo_operacional.add_edge("validar", "executar")
grafo_operacional.add_edge("executar", END)

app_operacional = grafo_operacional.compile()

### Visualização dos fluxos

O primeiro diagrama mostra a sequência estratégica. O segundo mostra o ciclo operacional com chamadas de ferramenta limitadas.

In [22]:
print("Modo estratégico:")
print(app_estrategico.get_graph().draw_mermaid())
print("\nModo operacional:")
print(app_operacional.get_graph().draw_mermaid())

Modo estratégico:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	observar(observar)
	planejar(planejar)
	decidir(decidir)
	validar(validar)
	executar(executar)
	__end__([<p>__end__</p>]):::last
	__start__ --> observar;
	decidir --> validar;
	observar --> planejar;
	planejar --> decidir;
	validar --> executar;
	executar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


Modo operacional:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	preparar(preparar)
	decidir(decidir)
	tools(tools<hr/><small><em>__tools = get_board_state, get_valid_moves, make_move, get_game_status</em></small>)
	fallback(fallback)
	validar(validar)
	executar(executar)
	__end__([<p>__end__</p>]):::last
	__start__ --> preparar;
	decidir -.-> __end__;
	decidir -.-> fallback;
	decidir -.-> tools;
	fallback --> validar;
	preparar --> decidir;
	tools -.-> 

In [23]:
def executar_turno_agente():
    if estado_jogo["jogador_atual"] is None:
        return None

    quantidade_anterior = len(estado_jogo["historico"])

    if MODO_AUTONOMIA == "estrategica":
        resultado = app_estrategico.invoke({"messages": []})
        solicitadas = resultado.get("ferramentas_solicitadas", [])
        executadas = resultado.get("ferramentas_executadas", [])
    else:
        resultado = app_operacional.invoke(
            {"messages": []},
            config={"recursion_limit": 20}
        )
        mensagens = resultado.get("messages", [])
        solicitadas = [
            chamada["name"]
            for mensagem in mensagens
            if isinstance(mensagem, AIMessage)
            for chamada in mensagem.tool_calls
        ]
        executadas = [
            mensagem.name
            for mensagem in mensagens
            if isinstance(mensagem, ToolMessage) and mensagem.name
        ]

        if (
            len(estado_jogo["historico"]) == quantidade_anterior + 1
            and not resultado.get("origem_decisao")
        ):
            resultado["origem_decisao"] = "llm_tool_call"

        if resultado.get("origem_decisao") == "fallback_heuristico":
            executadas.append("make_move")

    if len(estado_jogo["historico"]) != quantidade_anterior + 1:
        raise RuntimeError("O agente deve executar exatamente uma jogada.")

    jogada = estado_jogo["historico"][-1]
    origem = resultado.get("origem_decisao", "llm")
    jogada.update({
        "modo_autonomia": MODO_AUTONOMIA,
        "origem_decisao": origem,
        "ferramentas_solicitadas": solicitadas,
        "ferramentas_executadas": executadas
    })
    if resultado.get("erro_decisao"):
        jogada["erro_decisao"] = resultado["erro_decisao"]

    resultado["ferramentas_solicitadas"] = solicitadas
    resultado["ferramentas_executadas"] = executadas
    print("Modo:", MODO_AUTONOMIA)
    print("Origem da decisão:", origem)
    print("Jogada:", (jogada["row"], jogada["col"]))
    print("Ferramentas solicitadas:", solicitadas)
    print("Ferramentas executadas:", executadas)

    return resultado

## 7. Memória e planejamento

O histórico alimenta as decisões e o planejamento compara cada jogada com as respostas possíveis do adversário.

### 7.1 — Memória: histórico de jogadas

In [24]:
def obter_historico():
    historico = estado_jogo["historico"]

    if not historico:
        return "Nenhuma jogada realizada."

    jogadas = []
    for numero, jogada in enumerate(historico, start=1):
        jogador = "PRETO" if jogada["player"] == PRETO else "BRANCO"
        jogadas.append(
            f"{numero}. {jogador}: "
            f"linha {jogada['row']}, coluna {jogada['col']}"
        )

    return "\n".join(jogadas)

### 7.2 — Planejamento: avaliar as possíveis respostas

In [25]:
def planejar_jogadas(tabuleiro, jogador):
    analises = []

    for linha, coluna in movimentos_validos(tabuleiro, jogador):
        simulacao = [l.copy() for l in tabuleiro]
        fazer_jogada(simulacao, linha, coluna, jogador)

        pontuacao_atual = avaliar_tabuleiro(simulacao, jogador)
        respostas = movimentos_validos(simulacao, -jogador)
        pior_pontuacao = pontuacao_atual

        if respostas:
            pontuacoes = []
            for l, c in respostas:
                resposta = [linha.copy() for linha in simulacao]
                fazer_jogada(resposta, l, c, -jogador)
                pontuacoes.append(avaliar_tabuleiro(resposta, jogador))

            pior_pontuacao = min(pontuacoes)

        analises.append({
            "row": linha,
            "col": coluna,
            "score_now": pontuacao_atual,
            "worst_response": pior_pontuacao,
            "opponent_moves": len(respostas)
        })

    return analises

### 7.3 — Validação da jogada e origem da decisão

As decisões estruturadas e as chamadas de ferramenta são conferidas contra as jogadas legais. O histórico registra a modalidade, a origem da jogada e as ferramentas solicitadas e executadas para apoiar os experimentos.

## 8. Partida entre o agente LLM e o heurístico

A função usa o modo selecionado para as peças pretas e o agente heurístico para as brancas. Cada jogada registra sua origem e ferramentas para facilitar comparações posteriores.

In [26]:
def jogar_partida_completa():
    global estado_jogo
    estado_jogo = iniciar_partida()

    while estado_jogo["jogador_atual"] is not None:
        jogador = estado_jogo["jogador_atual"]
        quantidade_anterior = len(estado_jogo["historico"])

        if jogador == PRETO:
            executar_turno_agente()
        else:
            jogada = escolher_melhor_jogada(
                estado_jogo["tabuleiro"], jogador
            )
            if jogada is None:
                raise ValueError("Jogador atual sem movimentos válidos.")

            resultado = make_move.invoke({
                "row": jogada["row"],
                "col": jogada["col"]
            })
            if not resultado["success"]:
                raise RuntimeError(resultado["message"])

            estado_jogo["historico"][-1].update({
                "modo_autonomia": "heuristica",
                "origem_decisao": "heuristica",
                "ferramentas_solicitadas": [],
                "ferramentas_executadas": ["make_move"]
            })

        if len(estado_jogo["historico"]) != quantidade_anterior + 1:
            raise RuntimeError("Cada turno deve executar uma jogada.")

    return {
        "status": get_game_status.invoke({}),
        "historico": [
            jogada.copy() for jogada in estado_jogo["historico"]
        ]
    }

In [27]:
estado_jogo = iniciar_partida()

resultado = executar_turno_agente()

print("\n--- RESULTADO ---")
print("Jogada:", estado_jogo["historico"][-1])
print("Histórico:", estado_jogo["historico"])
print("Próximo jogador:", estado_jogo["jogador_atual"])
print("Campos retornados:", list(resultado.keys()))

assert len(estado_jogo["historico"]) == 1

print("Teste inicial aprovado!")

Modo: operacional
Origem da decisão: llm_tool_call
Jogada: (2, 3)
Ferramentas solicitadas: ['make_move']
Ferramentas executadas: ['make_move']

--- RESULTADO ---
Jogada: {'player': 1, 'row': 2, 'col': 3, 'modo_autonomia': 'operacional', 'origem_decisao': 'llm_tool_call', 'ferramentas_solicitadas': ['make_move'], 'ferramentas_executadas': ['make_move']}
Histórico: [{'player': 1, 'row': 2, 'col': 3, 'modo_autonomia': 'operacional', 'origem_decisao': 'llm_tool_call', 'ferramentas_solicitadas': ['make_move'], 'ferramentas_executadas': ['make_move']}]
Próximo jogador: -1
Campos retornados: ['messages', 'observacao', 'historico', 'planejamento', 'origem_decisao', 'erro_decisao', 'historico_inicial', 'chamadas_ferramenta', 'ferramentas_executadas', 'ferramentas_solicitadas']
Teste inicial aprovado!
